<a href="https://colab.research.google.com/github/shreyas-thirumale/rf-classification/blob/main/RF_Classification_new_interface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Dependencies

In [ ]:
import torch as t
from torch.utils.data import TensorDataset
from torch.utils.data import Dataset, DataLoader
import torchvision.datasets as datasets
from sklearn.model_selection import train_test_split
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import pickle
import numpy as np
import os
import base64

from itertools import cycle
import copy
from collections import defaultdict
from torch.utils.data import ConcatDataset

from google.colab import drive
drive.mount('/content/drive')


# Drone Audio Data Preparation


In [ ]:
!git clone https://github.com/saraalemadi/DroneAudioDataset.git

In [ ]:
import os
from pydub import AudioSegment
from pydub.utils import make_chunks
from google.colab import drive
# --- Configuration ---
# Set this to the duration of the Drone dataset samples (usually 1000ms)
CHUNK_LENGTH_MS = 1000

SOURCE_DIR = '/content/drive/MyDrive/ARL/Airport Noise Dataset/airport_no_human'
TARGET_DIR = '/content/DroneAudioDataset/Multiclass_Drone_Audio/bg noise'

os.makedirs(TARGET_DIR, exist_ok=True)

def slice_audio_files():
    file_list = [f for f in os.listdir(SOURCE_DIR) if f.endswith(('.wav', '.mp3'))]
    total_segments = 0

    print(f"Slicing {len(file_list)} files into {CHUNK_LENGTH_MS}ms chunks...")

    for filename in file_list:
        file_path = os.path.join(SOURCE_DIR, filename)

        try:
            audio = AudioSegment.from_file(file_path)

            # Divide audio into chunks
            chunks = make_chunks(audio, CHUNK_LENGTH_MS)

            for i, chunk in enumerate(chunks):
                # Only keep chunks that are exactly the right length
                # (discards the short 'tail' at the end of a file)
                if len(chunk) == CHUNK_LENGTH_MS:
                    output_name = f"{os.path.splitext(filename)[0]}_chunk{i}.wav"
                    chunk.export(os.path.join(TARGET_DIR, output_name), format="wav")
                    total_segments += 1

        except Exception as e:
            print(f"Error slicing {filename}: {e}")

    print(f"--- Process Finished ---")
    print(f"Created {total_segments} uniform segments in: {TARGET_DIR}")

slice_audio_files()

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import librosa
import numpy as np
from sklearn.model_selection import train_test_split

# --- Configuration ---
DATA_PATHS = {
    "mambo": "/content/DroneAudioDataset/Multiclass_Drone_Audio/membo_1",
    "bebop": "/content/DroneAudioDataset/Multiclass_Drone_Audio/bebop_1",
    "background": "/content/DroneAudioDataset/Multiclass_Drone_Audio/bg noise"
}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Custom Dataset Class ---
class DroneDataset(Dataset):
    def __init__(self, files, labels):
        self.files = files
        self.labels = labels

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        # Load and extract Mel-Spectrogram
        y, sr = librosa.load(self.files[idx], duration=1.0, sr=22050)
        if len(y) < 22050: y = np.pad(y, (0, 22050 - len(y)))

        spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        spec_db = librosa.power_to_db(spec, ref=np.max)

        # Normalize to [-1, 1] range for better convergence
        spec_db = (spec_db - np.mean(spec_db)) / np.std(spec_db)

        # Convert to tensor (Channel, Freq, Time)
        return torch.tensor(spec_db, dtype=torch.float32).unsqueeze(0), torch.tensor(self.labels[idx], dtype=torch.long)

# --- CNN Architecture ---
class DroneCNN(nn.Module):
    def __init__(self):
        super(DroneCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )
        # Assuming input is (128, 44) after melspectrogram/pooling
        # Linear layer size depends on input shape - 32x11 approx after 2 maxpools
        self.fc_layers = nn.Sequential(
            nn.Linear(64 * 32 * 11, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 3) # 3 classes: Mambo, Bebop, Background
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

# --- Data Preparation ---
all_files, all_labels = [], []
for label, (name, path) in enumerate(DATA_PATHS.items()):
    class_files = [os.path.join(path, f) for f in os.listdir(path) if f.endswith('.wav')][:800]
    all_files.extend(class_files)
    all_labels.extend([label] * len(class_files))

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(all_files, all_labels, test_size=0.2)

train_loader = DataLoader(DroneDataset(X_train_f, y_train_f), batch_size=32, shuffle=True)
test_loader = DataLoader(DroneDataset(X_test_f, y_test_f), batch_size=32)

# --- Training Loop ---
model = DroneCNN().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Starting Training...")
for epoch in range(10):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(DEVICE), target.to(DEVICE)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} completed.")

# --- Save the Model ---
torch.save(model.state_dict(), '/content/drive/MyDrive/ARL/drone_multi_classifier.pt')
print("✅ Model saved as drone_multi_classifier.pt")

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Setup Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Re-instantiate the Model Architecture
# Note: Ensure this class matches the DroneCNN class from your training script exactly
model = DroneCNN().to(DEVICE)

# 3. Load the Saved Weights
model.load_state_dict(torch.load('/content/drive/MyDrive/ARL/drone_multi_classifier.pt', map_location=DEVICE))
model.eval() # Set to evaluation mode (disables dropout)

print("✅ Model loaded successfully. Starting evaluation...")

# 4. Run Evaluation
all_preds = []
all_targets = []

with torch.no_grad(): # Disable gradient calculation for speed and memory
    for data, target in test_loader:
        data, target = data.to(DEVICE), target.to(DEVICE)

        outputs = model(data)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(target.cpu().numpy())

# 5. Calculate Accuracy
all_preds = np.array(all_preds)
all_targets = np.array(all_targets)
accuracy = (all_preds == all_targets).mean()

print(f"\n--- Overall Accuracy: {accuracy * 100:.2f}% ---\n")

# 6. Detailed Metrics
target_names = ['Mambo', 'Bebop', 'Airport Noise']
print("Classification Report:")
print(classification_report(all_targets, all_preds, target_names=target_names))

# RF Client

In [ ]:
def save_data(data_var,filename):
  with open(filename, 'wb') as file:
    # Use pickle.dump() to write the variable to the file
    pickle.dump(data_var, file)

def read_data(filename):
  with open(filename, 'rb') as f:
        # Load the object from the pickle file
        loaded_data = pickle.load(f)
  return loaded_data

class IQCNN(nn.Module):
    def __init__(self, num_classes=11):
        super(IQCNN, self).__init__()

        self.layer_dims=[]
        self.layers=nn.ModuleList()

        # Define convolutional layers here .......................................
        self.layers.append(nn.Conv1d(in_channels=2, out_channels=8, kernel_size=7, padding=3,bias=False))
        self.layers.append(nn.Conv1d(in_channels=8, out_channels=16, kernel_size=7, padding=3,bias=False))
        self.layers.append(nn.Conv1d(in_channels=16, out_channels=32, kernel_size=7, padding=3,bias=False))
        self.layers.append(nn.Conv1d(in_channels=32, out_channels=64, kernel_size=7, padding=3,bias=False))

        self.conv_num=len(self.layers)
        for i in range(self.conv_num):
          in_ch=self.layers[i].in_channels
          ker_sz=self.layers[i].kernel_size[0]
          self.layer_dims.append(in_ch*ker_sz)

        #Define linear layers here .....................................................
        self.layers.append(nn.Linear(64, 256,bias=False))
        self.layers.append(nn.Linear(256, num_classes,bias=False))

        for i in range(self.conv_num,len(self.layers)):
          self.layer_dims.append(self.layers[i].in_features)
        self.layer_dims.append(num_classes)
        # print(self.layer_dims)

        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)

        for i in range(self.conv_num):
          nn.init.xavier_uniform_(self.layers[i].weight)
        for i in range(self.conv_num,len(self.layers)-1):
            nn.init.kaiming_normal_(self.layers[i].weight, nonlinearity='relu')

        nn.init.kaiming_normal_(self.layers[i].weight, nonlinearity='linear')

    def forward(self, x):

        for i in range(self.conv_num):
          x=F.relu(self.layers[i](x))


        x = self.global_avg_pool(x)  # (batch_size, 64, 1)
        x = x.squeeze(-1)  # (batch_size, 64)

        for i in range(self.conv_num,len(self.layers)-1):
          x=F.relu(self.layers[i](x))
        x = self.layers[-1](x)  # (batch_size, output_shape)

        return x#F.log_softmax(x, dim=1)  # Use log_softmax for classification

def build_model(num_classes=2,model_name='IQCNN'):
  return IQCNN(num_classes=num_classes)#IQCNN(num_classes=11)

#.............................Client Class......................................

class Client:
  def __init__(self,dataset,num_classes=2,device='cpu'):
      self.device=device
      self.model = build_model(num_classes=num_classes).to(self.device)
      self.dataset =dataset
      self.dataloader = cycle(t.utils.data.DataLoader(self.dataset, batch_size=512, shuffle=True))
      self.cross_loss = nn.CrossEntropyLoss()
      self.optimizer = t.optim.Adam(self.model.parameters(), lr=0.001)

  def load_model(self,state_dict):
    self.model.load_state_dict(state_dict)

  def load_model_from_file(self,filename):
    state_dict=read_data(filename)
    self.load_model(state_dict)

  def save_model(self,filename):
    save_data(self.model.state_dict,filename)

  def client_loss(self,pred,y):
    return self.cross_loss(pred,y)

  def evaluate(self,test_loader):
    correct,total=0,0
    with t.no_grad():
        for data in test_loader:
            x, y = data
            x=x.to(self.device)
            y=y.to(self.device)
            output = self.model(x)
            for idx, i in enumerate(output):
                if t.argmax(i) == y[idx]:
                    correct +=1
                total +=1
    print(f'accuracy: {round(correct/total, 3)}')
    return round(correct/total, 3)


  def train_batch(self):
    x,y = next(self.dataloader)
    x=x.to(self.device)
    y=y.to(self.device)
    self.optimizer.zero_grad()
    pred = self.model(x)
    loss=self.client_loss(pred,y)
    loss.backward()
    self.optimizer.step()
    return loss.item()

  def train(self,test_loader,epochs=int(1e3),echo=True,eval_acc=False):
    running_loss=0
    for epoch in range(epochs):
      epoch_loss = self.train_batch()
      running_loss+=epoch_loss

      if echo:
        if epoch%int(epochs/10) ==0 and epoch!=0:
          print("epoch:"+str(epoch)+" loss:"+ str(running_loss/int(epochs/10)))
          running_loss=0
          if eval_acc:
            self.evaluate(test_loader)
        # if epoch%(echo*10)==0 and epoch!=0:
        #   self.evaluate()

    return running_loss/int(epochs/10)

def create_synthetic_data(noise_std,grid_sz,grid_step,num_datapoints=int(1e3)):
  seq_sz=128
  adversary_pwr=1
  num_classes=2 # 0: good 1: adversary

  # grid_dist=[np.sqrt(i**2+j**2) for i in range(grid_sz+1) for j in range(i+1)]
  # grid_step=0.5
  # grid_dist=[i*grid_step for i in range(int(grid_sz/grid_step))]
  grid_dist=[i for i in range(int(grid_sz/grid_step))]

  # Scales for Unit Power
  scale_qpsk = 1.0 / np.sqrt(2)
  scale_16qam = 1.0 / np.sqrt(10)

  mod_options = ['BPSK', 'QPSK', '16-QAM']
  label_dict={'BPSK':0, 'QPSK':1, '16-QAM':2}
  data=[]
  labels=[]

  for c in range(num_classes):
    for d in grid_dist:
      for _ in range(num_datapoints):
        p=np.random.uniform(0, 1)
        rx_i, rx_q = np.array([]), np.array([])
        bpsk_sig=(2*np.random.randint(0,2,size=seq_sz)-1) + 0j
        qpsk_sig= (2*np.random.randint(0,2,size=seq_sz)-1) + 1j*(2*np.random.randint(0,2,seq_sz)-1)
        qpsk_sig=scale_qpsk*qpsk_sig

        merge_vec=np.random.uniform(0,1,size=seq_sz)
        merge_vec=merge_vec<=p

        sig=(merge_vec)*bpsk_sig+(1-merge_vec)*qpsk_sig

        if c==1:
          mp = np.array([-3, -1, 1, 3])
          adv_sig= mp[np.random.randint(0,4,size=seq_sz)] + 1j*mp[np.random.randint(0,4,size=seq_sz)]
          adv_sig=scale_16qam
          if d!=0:
            sig=(adversary_pwr/d)*adv_sig+sig
          else:
            sig=adv_sig+sig

        noise = noise_std * (np.random.randn(seq_sz) +1j * np.random.randn(seq_sz))
        sig=sig+noise

        data.append(np.array([np.real(sig),np.imag(sig)]))
        labels.append(c)

  X=np.array(data)
  Y=np.array(labels)
  print(X.shape,Y.shape)

  X_train_ar, X_test_ar, y_train_ar, y_test_ar = train_test_split(X, Y, test_size=0.3, random_state=42, stratify=Y) # split the data (70% train, 30% test)
  print(X_train_ar.shape, X_test_ar.shape, y_train_ar.shape, y_test_ar.shape)

  X_train = t.from_numpy(X_train_ar).float()
  y_train = t.from_numpy(y_train_ar).long() # Use long for integer labels
  X_test = t.from_numpy(X_test_ar).float()
  y_test = t.from_numpy(y_test_ar).long()
  trainset = TensorDataset(X_train, y_train)
  testset = TensorDataset(X_test, y_test)

  batch_size = 512

  train_loader = t.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)
  test_loader = t.utils.data.DataLoader(testset, batch_size=10, shuffle=True)

  return train_loader,test_loader,trainset,testset


def generate_sensor_rf_sample(noise_std,p,c,d):
  """
  p: bpsk qpsk prob
  c: 1 adversary
  d: distance
  """
  seq_sz=128
  adversary_pwr=1
  # Scales for Unit Power
  scale_qpsk = 1.0 / np.sqrt(2)
  scale_16qam = 1.0 / np.sqrt(10)

  bpsk_sig=(2*np.random.randint(0,2,size=seq_sz)-1) + 0j
  qpsk_sig= (2*np.random.randint(0,2,size=seq_sz)-1) + 1j*(2*np.random.randint(0,2,seq_sz)-1)
  qpsk_sig=scale_qpsk*qpsk_sig

  merge_vec=np.random.uniform(0,1,size=seq_sz)
  merge_vec=merge_vec<=p

  sig=(merge_vec)*bpsk_sig+(1-merge_vec)*qpsk_sig

  if c==1:
    mp = np.array([-3, -1, 1, 3])
    adv_sig= mp[np.random.randint(0,4,size=seq_sz)] + 1j*mp[np.random.randint(0,4,size=seq_sz)]
    adv_sig=scale_16qam
    if d!=0:
      sig=(adversary_pwr/d)*adv_sig+sig
    else:
      sig=adv_sig+sig

  noise = noise_std * (np.random.randn(seq_sz) +1j * np.random.randn(seq_sz))
  sig=sig+noise

  sample=t.from_numpy(np.array([np.real(sig),np.imag(sig)])).to(device)
  sample=sample.to(t.float32)
  return sample.unsqueeze(0)

In [ ]:
train_loader,test_loader,trainset,testset=create_synthetic_data(noise_std=0.1,grid_sz=80,grid_step=4,num_datapoints=int(1e3))
device= 'cuda' if t.cuda.is_available() else 'cpu'
print(device)

In [ ]:
rf_client= Client(trainset,device=device)
rf_client.train(test_loader,epochs=int(3e3),echo=True,eval_acc=True)
rf_client.evaluate(test_loader)

In [ ]:
rf_client = read_data('/content/drive/MyDrive/ARL/sensor_client.pkl')
#save_data(rf_client,'/content/drive/MyDrive/ARL/sensor_client.pkl')

# Interface (Visual + RF)

In [ ]:
!pip install -qU gradio nest-asyncio mcp langchain-openai langchain-mcp-adapters langgraph langchain-ollama

In [ ]:
!unzip /content/drive/MyDrive/ARL/drone_demo_dataset.zip -d /content/demo_images

In [ ]:
  #@title { vertical-output: true}

import torch
import gc
import os
import io
import time
import threading
import subprocess
import asyncio
from collections import Counter

import gradio as gr
import numpy as np
import matplotlib.pyplot as plt
import nest_asyncio
from io import BytesIO
from PIL import Image
from mcp.server.fastmcp import FastMCP
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_ollama import ChatOllama

import base64
import random
from tabulate import tabulate

import os
import random
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')


os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()

univ_hop_dur = 0.8
univ_sph = 8


IMAGE_PATH = "/content/drive/MyDrive/ARL/dca.png"
DRONE_IMAGE_PATH = "/content/drive/MyDrive/ARL/drone.png"
IS_DRONE_IN_RANGE = False
IS_DRONE_AUDIO_RANGE = False
IMAGE_WIDTH_PX = 500
IMAGE_HEIGHT_PX = 693

SENSOR_RADIUS_PX = 50
NUM_SENSORS = 25
DRONE_RADIUS_PX = 10  # since the drone is 20x20

#convert white background in drone to alpha
from PIL import Image
import os
import base64
from io import BytesIO



def load_drone_as_data_uri(path, white_thresh=200):
    if not os.path.exists(path):
        return ""

    img = Image.open(path).convert("RGBA")
    data = img.getdata()

    new_data = []
    for r, g, b, a in data:
        if r >= white_thresh and g >= white_thresh and b >= white_thresh:
            new_data.append((255, 255, 255, 0))
        else:
            new_data.append((r, g, b, a))

    img.putdata(new_data)

    buffer = BytesIO()
    img.save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{encoded}"

#Drone Movement Parameters
CURRENT_POS = (
    random.randint(DRONE_RADIUS_PX, IMAGE_WIDTH_PX - DRONE_RADIUS_PX),
    random.randint(DRONE_RADIUS_PX, IMAGE_HEIGHT_PX - DRONE_RADIUS_PX),
)
LAST_MOVE_TIME = None

SENSORS = [
    (235, 37),
    (190, 89),
    (167, 135),
    (190, 221),
    (165, 287),
    (179, 350),
    (164, 417),
    (106, 462),
    (105, 550),
    (161, 593),
    (226, 633),
    (307, 650),
    (383, 617),
    (403, 551),
    (420, 480),
    (431, 421),
    (435, 345),
    (429, 285),
    (412, 224),
    (394, 177),
    (373, 128),
    (332, 86),
    (305, 38)
]

RF_VISIBILITY_RNG=80
GRID_STEP=4
SENSOR_PROBS=np.random.uniform(0,1,size=len(SENSORS))
SENSOR_MODEL=read_data('/content/drive/MyDrive/ARL/sensor_client.pkl')

# --- 1. CONFIGURATION ---
class SystemState:
    def __init__(self):
        self.fs = 4000
        self.chunk_duration = 4.0
        self.running = False
        self.current_waveform = np.array([])
        self.gt_iq = []
        self.gt_mod = []
        self.gt_fc = []
        self.demod = []
        self.cur_analysis = None
        self.cur_spec_base64 = None

        self.profiles = {
            "S1 (Long Range)": {
                "freqs": [10, 20],
                "mod": "BPSK",
                "color": "#00CCFF",
                "desc": "Robust / Low Rate"
            },
            "S2 (Standard)": {
                "freqs": [30, 40],
                "mod": "QPSK",
                "color": "#00FF00",
                "desc": "Standard Link"
            },
            "S3 (THREAT)": {
                "freqs": [50, 60, 70],
                "mod": "16-QAM",
                "color": "#FF0000",
                "desc": "High Speed Burst"
            }
        }
        
        self.latest_snapshot = {
            "timestamp": None,
            "drone_pos": None,
            "is_threat": None,
            "rf": {
                "triggered_sensors": [],
                "detected": False
            },
            "audio": {
                "confidence": 0.0,
                "detected": False
            },
            "vision": {
                "confidence": 0.0,
                "detected": False
            },
            "fused": {
                "drone_present": False,
                "signal_threat": False,
                "overall_threat": False,
                "overall_confidence": 0.0,
                "votes": 0
            }
        }



sys_state = SystemState()




# --- 4. STATUS CIRCLE + VIS PANEL ---

#helper functions
def point_to_segment_distance(px, py, x1, y1, x2, y2):
        dx = x2 - x1
        dy = y2 - y1

        if dx == 0 and dy == 0:
            return ((px - x1) ** 2 + (py - y1) ** 2) ** 0.5

        t = ((px - x1) * dx + (py - y1) * dy) / (dx * dx + dy * dy)
        t = max(0.0, min(1.0, t))

        closest_x = x1 + t * dx
        closest_y = y1 + t * dy

        return ((px - closest_x) ** 2 + (py - closest_y) ** 2) ** 0.5

# --- RANDOM PATH GENERATION IN PIXELS ---
def rand_pos():
    step = 40  # max movement per refresh, in pixels
    new_x = CURRENT_POS[0] + random.randint(-step, step)
    new_y = CURRENT_POS[1] + random.randint(-step, step)

    new_x = max(DRONE_RADIUS_PX, min(new_x, IMAGE_WIDTH_PX - DRONE_RADIUS_PX))
    new_y = max(DRONE_RADIUS_PX, min(new_y, IMAGE_HEIGHT_PX - DRONE_RADIUS_PX))
    return (new_x, new_y)

def current_threat_active():
    if not sys_state.gt_mod:
        return np.random.binomial(n=1, p=0.8, size=1)[0]
    return "16-QAM" in sys_state.gt_mod


def drone_movement_and_detection():
    global CURRENT_POS, LAST_MOVE_TIME
    global IS_DRONE_IN_RANGE, IS_DRONE_AUDIO_RANGE

    now = time.time()

    if LAST_MOVE_TIME is None:
        dt = 2.0
    else:
        dt = now - LAST_MOVE_TIME

    LAST_MOVE_TIME = now

    p1 = CURRENT_POS
    p2 = rand_pos()
    CURRENT_POS = p2

    is_threat = current_threat_active()

    sensor_states = []
    any_triggered = False
    audio_triggered = False

    sys_state.cur_analysis=[]

    for idx,sp in enumerate(SENSORS):
        sx, sy=sp
        dist = point_to_segment_distance(
            sx, sy,
            p1[0], p1[1],
            p2[0], p2[1]
        )


        # triggered = is_threat and (dist <= SENSOR_RADIUS_PX)
        triggered = False
        if dist<=RF_VISIBILITY_RNG:
          d=int(dist/GRID_STEP)
          sample=generate_sensor_rf_sample(0.1,SENSOR_PROBS[idx],is_threat,d)
          pred=torch.argmax(SENSOR_MODEL.model(sample))
          triggered=(pred==1)

        if triggered:
          sys_state.cur_analysis.append(idx)



        a_triggered = (dist <= SENSOR_RADIUS_PX)

        any_triggered = any_triggered or triggered
        audio_triggered = audio_triggered or a_triggered

        sensor_states.append({
            "x": sx,
            "y": sy,
            "triggered": triggered,
            "audio_triggered": a_triggered,
        })

    IS_DRONE_IN_RANGE = any_triggered
    IS_DRONE_AUDIO_RANGE = audio_triggered

    return {
        "p1": p1,
        "p2": p2,
        "dt": dt,
        "is_threat": is_threat,
        "sensor_states": sensor_states,
    }

def build_status_circle_html():
    movement = drone_movement_and_detection()
    movement_duration = max(0.05, movement["dt"])

    p1 = movement["p1"]
    p2 = movement["p2"]
    is_threat = movement["is_threat"]
    sensor_states = movement["sensor_states"]

    drone_color = "#ff2b2bff" if is_threat else "#22c55e00"
    animation_name = f"moveDot{random.randint(0, 100000)}"

    # --- LOAD IMAGES ---
    #background
    img_src = ""
    if os.path.exists(IMAGE_PATH):
        with open(IMAGE_PATH, "rb") as f:
            encoded = base64.b64encode(f.read()).decode("utf-8")
        ext = IMAGE_PATH.split(".")[-1].lower()
        if ext == "jpg":
            ext = "jpeg"
        img_src = f"data:image/{ext};base64,{encoded}"
    #drone
    drone_img_src = load_drone_as_data_uri(DRONE_IMAGE_PATH)

    #drone rendering
    drone_filter = (
      "brightness(0) invert(0.6)"  # grey
      if not is_threat else
      "brightness(0) saturate(100%) invert(20%) sepia(100%) saturate(6000%) hue-rotate(0deg) brightness(1.2)"
    )

    # --- SENSOR RENDERING ---
    sensor_html = []
    for sensor in sensor_states:
        sx = sensor["x"]
        sy = sensor["y"]
        triggered = sensor["triggered"]

        sensor_color = (
            "rgba(255, 165, 0, 0.16)"
            if triggered else
            "rgba(100, 180, 255, 0.10)"
        )
        sensor_border = (
            "rgba(255, 180, 80, 0.40)"
            if triggered else
            "rgba(120, 190, 255, 0.22)"
        )

        sensor_html.append(f"""
            <div style="
                position: absolute;
                left: {sx}px;
                top: {sy}px;
                width: {2 * SENSOR_RADIUS_PX}px;
                height: {2 * SENSOR_RADIUS_PX}px;
                transform: translate(-50%, -50%);
                border-radius: 50%;
                background: {sensor_color};
                border: 1.5px solid {sensor_border};
                box-shadow: 0 0 12px {sensor_color};
                pointer-events: none;
            "></div>
        """)

    sensors_html = "\n".join(sensor_html)

    return f"""
    <style>
    @keyframes {animation_name} {{
        0%   {{ left: {p1[0]}px; top: {p1[1]}px; }}
        100% {{ left: {p2[0]}px; top: {p2[1]}px; }}
    }}
    </style>

    <div style="
        height: 100%;
        min-height: {IMAGE_HEIGHT_PX + 32}px;
        display: flex;
        align-items: center;
        justify-content: center;
        background: #0f1117;
        border-radius: 18px;
        border: 1px solid #2a2f3a;
        padding: 16px;
        box-sizing: border-box;
    ">
        <div style="
            position: relative;
            width: {IMAGE_WIDTH_PX}px;
            height: {IMAGE_HEIGHT_PX}px;
            min-width: {IMAGE_WIDTH_PX}px;
            max-width: {IMAGE_WIDTH_PX}px;
            min-height: {IMAGE_HEIGHT_PX}px;
            max-height: {IMAGE_HEIGHT_PX}px;
            overflow: hidden;
            border-radius: 18px;
            flex-shrink: 0;
        ">

            <img src="{img_src}" style="
                width: {IMAGE_WIDTH_PX}px;
                height: {IMAGE_HEIGHT_PX}px;
                min-width: {IMAGE_WIDTH_PX}px;
                max-width: {IMAGE_WIDTH_PX}px;
                min-height: {IMAGE_HEIGHT_PX}px;
                max-height: {IMAGE_HEIGHT_PX}px;
                object-fit: fill;
                display: block;
                border-radius: 18px;
            ">

            {sensors_html}

            <img src="{drone_img_src}" style="
                position: absolute;
                left: {p1[0]}px;
                top: {p1[1]}px;
                width: {2 * DRONE_RADIUS_PX * 2}px;
                height: {2 * DRONE_RADIUS_PX * 2}px;
                transform: translate(-50%, -50%);
                animation: {animation_name} {movement_duration}s linear forwards;

                filter: {drone_filter};
                opacity: 0.95;

                pointer-events: none;

            ">

        </div>
    </div>
    """



def build_blank_visual_panel():
    return """
    <div style="
        width: 100%;
        aspect-ratio: 1 / 1;
        min-height: 320px;
        background: #0f1117;
        border: 2px dashed #3b4252;
        border-radius: 18px;
        display: flex;
        align-items: center;
        justify-content: center;
        color: #8b949e;
        font-size: 20px;
        font-weight: 600;
    ">
        Visualization Output
    </div>
    """


# ---- CHATBOT -----
import nest_asyncio
import threading
import asyncio
from mcp.server.fastmcp import FastMCP
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage

nest_asyncio.apply()

mcp = FastMCP("RF_Analysis_Assistant")
class DroneCNN(nn.Module):
    def __init__(self):
        super(DroneCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )
        # Assuming input is (128, 44) after melspectrogram/pooling
        # Linear layer size depends on input shape - 32x11 approx after 2 maxpools
        self.fc_layers = nn.Sequential(
            nn.Linear(64 * 32 * 11, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 3) # 3 classes: Mambo, Bebop, Background
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

def get_prediction_confidence(signal, sr):
    # 1. Pre-process signal to match training format
    if len(signal) < sr:
        signal = np.pad(signal, (0, int(sr - len(signal))))

    spec = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=128)
    spec_db = librosa.power_to_db(spec, ref=np.max)

    # 2. Normalize (Z-score) as done in training
    spec_db = (spec_db - np.mean(spec_db)) / (np.std(spec_db) + 1e-6)

    # 3. Convert to Tensor (B, C, H, W)
    input_tensor = torch.tensor(spec_db, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)

    # 4. Predict
    with torch.no_grad():
        output = model(input_tensor)
        probabilities = torch.nn.functional.softmax(output, dim=1)

    # Probabilities for [Mambo, Bebop, Airport]
    # Drone confidence = Mambo % + Bebop %
    mambo_prob = probabilities[0][0].item()
    bebop_prob = probabilities[0][1].item()
    drone_conf = (mambo_prob + bebop_prob) * 100

    return drone_conf


@mcp.tool()
def check_current_rf_status() -> str:
    """Reads the current RF environment variables including modulation and frequency hopping."""
    if sys_state.cur_analysis:
        # Prevent type error: cur_analysis is assigned as a list of integers during movement
        if isinstance(sys_state.cur_analysis, list):
          return f"Anomalous RF transmissions detected at sensor indices: {sys_state.cur_analysis}"
    return "No RF analysis available. The operator has not scanned a signal yet."
# # end rf tool...............................

# Vision Tool begins ============================================================================================================

_cached_vision_model = None
global_visual_html = None

@mcp.tool()
def check_current_visual_status() -> str:
    """
    Randomly samples an image from the visual sensor feeds, runs the ResNet50
    vision model, and returns whether a drone is present or not.
    Use this tool whenever the user asks to check cameras or look for drones visually.
    """
    global _cached_vision_model, global_visual_html
    _DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Initialize Model (Only runs once)
    if _cached_vision_model is None:
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, 1)
        # MUST MATCH YOUR SAVED WEIGHTS PATH
        checkpoint = torch.load('/content/drive/MyDrive/ARL/resnet50_drone_weights.pth', map_location=_DEVICE)

        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            model.load_state_dict(checkpoint)

        model.to(_DEVICE)
        model.eval()
        _cached_vision_model = model

    # 2. Pick Random Image
    demo_dir = Path('/content/demo_images')
    selected_category = 'drones' if IS_DRONE_AUDIO_RANGE else random.choice(['birds', 'planes'])
    all_images = list((demo_dir / selected_category).glob('*.jpg'))

    if not all_images:
        return "Error: Camera feed offline. No images found."

    img_path = random.choice(all_images)

    # 3. Preprocess & Infer
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    img = Image.open(img_path).convert('RGB')
    input_tensor = transform(img).unsqueeze(0).to(_DEVICE)

    with torch.no_grad():
        output = _cached_vision_model(input_tensor)
        prob = torch.sigmoid(output).item()

    is_drone = prob >= 0.5
    classification = "DRONE DETECTED" if is_drone else "CLEAR (Bird/Plane)"
    color = "#ef4444" if is_drone else "#22c55e" # Red for drone, Green for clear

    # 4. Generate the UI HTML secretly for Gradio
    buffered = BytesIO()
    img.save(buffered, format="JPEG")
    img_str = base64.b64encode(buffered.getvalue()).decode()

    global_visual_html = f"""
    <div style="position: relative; width: 100%; aspect-ratio: 1/1; background: #0f1117; border-radius: 18px; overflow: hidden; border: 2px solid {color};">
        <div style="position: absolute; top: 0; left: 0; right: 0; background: rgba(0,0,0,0.8); color: {color}; padding: 8px; text-align: center; font-family: monospace; font-weight: bold; z-index: 10;">
            OPTICAL SENSOR: {classification} ({prob:.1%} Conf)
        </div>
        <img src="data:image/jpeg;base64,{img_str}" style="width: 100%; height: 100%; object-fit: contain; padding-top: 35px;">
    </div>
    """

    # 5. Return text to the Chatbot Agent
    return f"Visual Analysis Complete. Result: {classification}. Confidence: {prob:.4f}."

# Vision Tool Ends =======================================================================================================

@mcp.tool()
def check_current_audio_status() -> str:
    """Checks audio signals to detect whether a drone is present."""
    return "Drone!"




# # end audio tool ............................................................

def start_server():
    print("Starting MCP Server...")
    mcp.run(transport="sse")

server_thread = threading.Thread(target=start_server, name="mcp_server_thread", daemon=True)
server_thread.start()

!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import threading
import subprocess
import time

def run_ollama():
    subprocess.run(["ollama", "serve"])

ollama_thread = threading.Thread(target=run_ollama, daemon=True)
ollama_thread.start()

print("✅ Ollama server is running!")
!ollama pull llama3.2

import time

time.sleep(5)

from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

time.sleep(5)

SYSTEM_PROMPT = (
    "You are an RF engineering assistant in the US army. "
    "You have access to two tools:\n"
    "1. `check_current_rf_status`: Analyzes and provides sensors where anomalous rf transmission were detected.\n"
    "2. `check_current_audio_status`: Checks audio signals to detect whether a drone is present. Use this whenever the user asks about audio, acoustics, or drone presence.\n\n"
    "3. `check_current_visual_status`: Takes a picture/photo of the current environment using a camera, and identifies whether a drone is present or not."
    "Based on the results from these tools, your task is to answer the questions so that even a civilian would understand what is going on."
)

global_agent = None


async def get_agent():
    global global_agent
    if global_agent is None:
        client = MultiServerMCPClient({
            "research": {
                "transport": "sse",
                "url": "http://localhost:8000/sse"
            }
        })
        mcp_tools = await client.get_tools()
        global_agent = create_react_agent(llm, mcp_tools)
    return global_agent


async def chat_fn(message, history):
    if not message:
        return "Please enter a question."

    try:
        agent = await get_agent()
        messages = [SystemMessage(content=SYSTEM_PROMPT)]

        if history:
            for msg in history:
                if msg.get("role") == "user":
                    messages.append(HumanMessage(content=msg.get("content", "")))
                elif msg.get("role") == "assistant":
                    messages.append(AIMessage(content=msg.get("content", "")))

        # if sys_state.cur_analysis:
        #     rft_prompt = (
        #         f"\nSTART_RFT Detected modulation is {sys_state.cur_analysis['mod'][0]}. "
        #         f"Detected frequency hopping sequence is {str(sys_state.cur_analysis['freq_hop'])}. END_RFT"
        #     )
        #     messages.append(HumanMessage(content=message + rft_prompt))
        # else:
        #     messages.append(HumanMessage(content=message))
        messages.append(HumanMessage(content=message))
        response = await agent.ainvoke({"messages": messages})
        return response["messages"][-1].content

    except Exception as e:
        return f"Error connecting to agent: {str(e)}"

def build_sensor_visual_panel(sensor_results, is_detected, hit_count, audio_path=None):
    """Generates the HTML for the Right-Hand visualization panel."""

    # 1. Decision Banner
    color = "#ff4b4b" if is_detected else "#4ade80"
    status_text = "🚨 DRONE DETECTED" if is_detected else "✅ ZONE CLEAR"

    # 2. Table to HTML
    table_html = tabulate(sensor_results, headers=["Sensor ID", "Dist", "Conf %"], tablefmt="unsafehtml")

    # 3. Audio Player (Optional: see note below)
    audio_html = ""
    # Note: In a real Gradio app, you'd save the attenuated_y to a temp .wav file
    # and pass the path here to display a <audio> tag.

    html = f"""
    <div style="background: #1a1c23; padding: 15px; border-radius: 12px; border: 1px solid #2a2f3a; color: white; font-family: sans-serif;">
        <div style="background: {color}; padding: 10px; border-radius: 8px; text-align: center; font-weight: bold; margin-bottom: 15px;">
            {status_text} ({hit_count} Hits)
        </div>
        <div style="max-height: 250px; overflow-y: auto; font-size: 0.85em;">
            {table_html}
        </div>
        <div style="margin-top: 15px; font-size: 0.8em; color: #888;">
            *Signal processed from Closest Sensor
        </div>
    </div>
    <style>
        table {{ width: 100%; border-collapse: collapse; }}
        th {{ text-align: left; border-bottom: 1px solid #444; padding: 5px; color: #aaa; }}
        td {{ padding: 5px; border-bottom: 1px solid #222; }}
    </style>
    """
    return html


async def respond(message, chat_history):
    if chat_history is None: chat_history = []

    # Default blank panel
    vis_html = build_blank_visual_panel()

    # try:
    #     agent = await get_agent()
    #     messages = [SystemMessage(content=SYSTEM_PROMPT)]
    #     for msg in chat_history:
    #         messages.append(AIMessage(content=msg["content"]) if msg["role"] == "assistant" else HumanMessage(content=msg["content"]))

    #     messages.append(HumanMessage(content=message))
    #     response = await agent.ainvoke({"messages": messages})
    #     bot_message = response["messages"][-1].content

    #     # CHECK IF A SPECTROGRAM WAS GENERATED
    #     if hasattr(sys_state, 'cur_spec_base64') and sys_state.cur_spec_base64:
    #         vis_html = f"""
    #         <div style="width: 100%; aspect-ratio: 1/1; background: #0f1117; border-radius: 18px; overflow: hidden; border: 1px solid #2a2f3a;">
    #             <img src="data:image/png;base64,{sys_state.cur_spec_base64}" style="width: 100%; height: 100%; object-fit: contain;">
    #         </div>
    #         """
    #         sys_state.cur_spec_base64 = None
    #     else:
    #         vis_html = build_blank_visual_panel()

    # except Exception as e:
    #     bot_message = f"Agent Error: {str(e)}"

    # chat_history.append({"role": "user", "content": message})
    # chat_history.append({"role": "assistant", "content": bot_message})

    # # IMPORTANT: Returning 3 values to match msg.submit outputs
    # return "", chat_history, vis_html

    try:
        agent = await get_agent()
        messages = [SystemMessage(content=SYSTEM_PROMPT)]
        for msg in chat_history:
            messages.append(AIMessage(content=msg["content"]) if msg["role"] == "assistant" else HumanMessage(content=msg["content"]))

        messages.append(HumanMessage(content=message))

        # 1. The agent thinks, decides to use the tool, and the tool sets global_visual_html
        response = await agent.ainvoke({"messages": messages})
        bot_message = response["messages"][-1].content

        # 2. CHECK WHICH SENSOR WAS TRIGGERED FOR THE UI
        global global_visual_html

        if global_visual_html:
            # The Vision Tool was used!
            vis_html = global_visual_html
            global_visual_html = None # Reset for next question

        elif hasattr(sys_state, 'cur_spec_base64') and sys_state.cur_spec_base64:
            # The RF Tool was used!
            vis_html = f"""
            <div style="width: 100%; aspect-ratio: 1/1; background: #0f1117; border-radius: 18px; overflow: hidden; border: 1px solid #2a2f3a;">
                <img src="data:image/png;base64,{sys_state.cur_spec_base64}" style="width: 100%; height: 100%; object-fit: contain;">
            </div>
            """
            sys_state.cur_spec_base64 = None

        else:
            # No sensors used, show blank square
            vis_html = build_blank_visual_panel()

    except Exception as e:
        bot_message = f"Agent Error: {str(e)}"
        vis_html = build_blank_visual_panel()

    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": bot_message})

    return "", chat_history, vis_html

# --- UI CONTROL FUNCTIONS ---
def refresh_ui(noise):

    status_html = build_status_circle_html()
    visual_html = build_blank_visual_panel()

    return status_html, visual_html, "Monitoring..."


def stream(noise):
    sys_state.running = True
    while sys_state.running:
        status_html, visual_html, status_text = refresh_ui(noise)
        yield status_html, visual_html, status_text
        time.sleep(2.0)

def pause():
    sys_state.running = False
    return "Paused"

# --- INTERFACE ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📡 RF Anomaly Detection Dashboard")

    with gr.Row():
        # LEFT: status circle only
        with gr.Column(scale=1):
            status_circle = gr.HTML(build_status_circle_html())

            noise = gr.Slider(0, 0.5, 0.1, label="Noise")
            btn_play = gr.Button("▶ Start", variant="primary")
            btn_pause = gr.Button("⏸ Pause", variant="secondary")
            out = gr.HTML("Ready")

        # RIGHT: chatbot on top, blank square below
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(label="ChatBot", height=320)
            msg = gr.Textbox(label="Your Message")
            clear = gr.Button("Clear Memory")
            visual_panel = gr.HTML(build_blank_visual_panel())

    btn_play.click(stream, [noise], [status_circle, visual_panel, out])
    btn_pause.click(pause, None, out)

    msg.submit(respond, [msg, chatbot], [msg, chatbot, visual_panel])
    clear.click(lambda: [], None, chatbot, queue=False)



if __name__ == "__main__":
    demo.queue().launch(debug=True)

# Interface (Audio +RF)

In [ ]:
class DroneCNN(nn.Module):
    def __init__(self):
        super(DroneCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(64 * 32 * 11, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

def get_prediction_confidence(signal, sr):
    if len(signal) < sr:
        signal = np.pad(signal, (0, int(sr - len(signal))))

    spec = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=128)
    spec_db = librosa.power_to_db(spec, ref=np.max)
    spec_db = (spec_db - np.mean(spec_db)) / (np.std(spec_db) + 1e-6)

    input_tensor = torch.tensor(spec_db, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        output = model(input_tensor)
        probabilities = torch.nn.functional.softmax(output, dim=1)

    mambo_prob = probabilities[0][0].item()
    bebop_prob = probabilities[0][1].item()
    drone_conf = (mambo_prob + bebop_prob) * 100

    return drone_conf
model = DroneCNN().to(DEVICE)
# Ensure model is loaded (Assumes file exists as per your code)
if os.path.exists('/content/drive/MyDrive/ARL/drone_multi_classifier.pt'):
    model.load_state_dict(torch.load('/content/drive/MyDrive/ARL/drone_multi_classifier.pt', map_location=DEVICE))
model.eval()

In [ ]:
!pip install -qU gradio nest-asyncio mcp langchain-openai langchain-mcp-adapters langgraph langchain-ollama

In [ ]:
import torch
import torch.nn as nn
import gc
import os
import io
import time
import threading
import subprocess
import asyncio
import librosa
from collections import Counter
import tempfile
import soundfile as sf

import gradio as gr
import numpy as np
import matplotlib.pyplot as plt
import nest_asyncio
from io import BytesIO
from PIL import Image
from mcp.server.fastmcp import FastMCP
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_ollama import ChatOllama

import base64
import random
from tabulate import tabulate

from google.colab import drive
drive.mount('/content/drive')


os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()

univ_hop_dur = 0.8
univ_sph = 8


IMAGE_PATH = "/content/drive/MyDrive/ARL/dca.png"
DRONE_IMAGE_PATH = "/content/drive/MyDrive/ARL/drone.png"
IS_DRONE_IN_RANGE = False
IS_DRONE_AUDIO_RANGE = False
IMAGE_WIDTH_PX = 500
IMAGE_HEIGHT_PX = 693

SENSOR_RADIUS_PX = 50
NUM_SENSORS = 25
DRONE_RADIUS_PX = 10  # since the drone is 20x20

def load_drone_as_data_uri(path, white_thresh=200):
    if not os.path.exists(path):
        return ""

    img = Image.open(path).convert("RGBA")
    data = img.getdata()

    new_data = []
    for r, g, b, a in data:
        if r >= white_thresh and g >= white_thresh and b >= white_thresh:
            new_data.append((255, 255, 255, 0))
        else:
            new_data.append((r, g, b, a))

    img.putdata(new_data)

    buffer = BytesIO()
    img.save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{encoded}"

#Drone Movement Parameters
CURRENT_POS = (
    random.randint(DRONE_RADIUS_PX, IMAGE_WIDTH_PX - DRONE_RADIUS_PX),
    random.randint(DRONE_RADIUS_PX, IMAGE_HEIGHT_PX - DRONE_RADIUS_PX),
)
LAST_MOVE_TIME = None

SENSORS = [
    (235, 37), (190, 89), (167, 135), (190, 221), (165, 287),
    (179, 350), (164, 417), (106, 462), (105, 550), (161, 593),
    (226, 633), (307, 650), (383, 617), (403, 551), (420, 480),
    (431, 421), (435, 345), (429, 285), (412, 224), (394, 177),
    (373, 128), (332, 86), (305, 38)
]

RF_VISIBILITY_RNG=80
GRID_STEP=4
SENSOR_PROBS=np.random.uniform(0,1,size=len(SENSORS))
SENSOR_MODEL=read_data('/content/drive/MyDrive/ARL/sensor_client.pkl')

# --- 1. CONFIGURATION ---
class SystemState:
    def __init__(self):
        self.fs = 4000
        self.chunk_duration = 4.0
        self.running = False
        self.current_waveform = np.array([])
        self.gt_iq = []
        self.gt_mod = []
        self.gt_fc = []
        self.demod = []
        self.cur_analysis = None
        self.cur_spec_base64 = None

        self.profiles = {
            "S1 (Long Range)": {
                "freqs": [10, 20],
                "mod": "BPSK",
                "color": "#00CCFF",
                "desc": "Robust / Low Rate"
            },
            "S2 (Standard)": {
                "freqs": [30, 40],
                "mod": "QPSK",
                "color": "#00FF00",
                "desc": "Standard Link"
            },
            "S3 (THREAT)": {
                "freqs": [50, 60, 70],
                "mod": "16-QAM",
                "color": "#FF0000",
                "desc": "High Speed Burst"
            }
        }
        self.latest_snapshot = {
            "timestamp": None,
            "drone_pos": None,
            "is_threat": None,
            "rf": {
                "triggered_sensors": [],
                "detected": False
            },
            "audio": {
                "confidence": 0.0,
                "detected": False
            },
            "vision": {
                "confidence": 0.0,
                "detected": False
            },
            "fused": {
                "drone_present": False,
                "signal_threat": False,
                "overall_threat": False,
                "overall_confidence": 0.0,
                "votes": 0
            }
        }

sys_state = SystemState()

# --- 4. STATUS CIRCLE + VIS PANEL ---
def point_to_segment_distance(px, py, x1, y1, x2, y2):
        dx = x2 - x1
        dy = y2 - y1

        if dx == 0 and dy == 0:
            return ((px - x1) ** 2 + (py - y1) ** 2) ** 0.5

        t = ((px - x1) * dx + (py - y1) * dy) / (dx * dx + dy * dy)
        t = max(0.0, min(1.0, t))

        closest_x = x1 + t * dx
        closest_y = y1 + t * dy

        return ((px - closest_x) ** 2 + (py - closest_y) ** 2) ** 0.5

# --- RANDOM PATH GENERATION IN PIXELS ---
def rand_pos():
    step = 40
    new_x = CURRENT_POS[0] + random.randint(-step, step)
    new_y = CURRENT_POS[1] + random.randint(-step, step)

    new_x = max(DRONE_RADIUS_PX, min(new_x, IMAGE_WIDTH_PX - DRONE_RADIUS_PX))
    new_y = max(DRONE_RADIUS_PX, min(new_y, IMAGE_HEIGHT_PX - DRONE_RADIUS_PX))
    return (new_x, new_y)

def current_threat_active():
    if not sys_state.gt_mod:
        return np.random.binomial(n=1, p=0.8, size=1)[0]
    return "16-QAM" in sys_state.gt_mod

def drone_movement_and_detection():
    global CURRENT_POS, LAST_MOVE_TIME
    global IS_DRONE_IN_RANGE, IS_DRONE_AUDIO_RANGE

    now = time.time()

    if LAST_MOVE_TIME is None:
        dt = 2.0
    else:
        dt = now - LAST_MOVE_TIME

    LAST_MOVE_TIME = now

    p1 = CURRENT_POS
    p2 = rand_pos()
    CURRENT_POS = p2

    is_threat = current_threat_active()

    sensor_states = []
    any_triggered = False
    audio_triggered = False

    sys_state.cur_analysis = []

    for idx, sp in enumerate(SENSORS):
        sx, sy = sp
        dist = point_to_segment_distance(sx, sy, p1[0], p1[1], p2[0], p2[1])

        triggered = False
        if dist <= RF_VISIBILITY_RNG:
          d = int(dist/GRID_STEP)
          sample = generate_sensor_rf_sample(0.1, SENSOR_PROBS[idx], is_threat, d)
          pred = torch.argmax(SENSOR_MODEL.model(sample))
          triggered = (pred == 1)

        if triggered:
          sys_state.cur_analysis.append(idx)

        a_triggered = (dist <= SENSOR_RADIUS_PX)

        any_triggered = any_triggered or triggered
        audio_triggered = audio_triggered or a_triggered

        sensor_states.append({
            "x": sx,
            "y": sy,
            "triggered": triggered,
            "audio_triggered": a_triggered,
        })

    IS_DRONE_IN_RANGE = any_triggered
    IS_DRONE_AUDIO_RANGE = audio_triggered

    # --- Audio inference ---
    SR = 22050
    audio_confidence = 0.0
    if audio_triggered:
        DRONE_DIRS = [
            "/content/DroneAudioDataset/Multiclass_Drone_Audio/membo_1",
            "/content/DroneAudioDataset/Multiclass_Drone_Audio/bebop_1"
        ]
        AIRPORT_DIR = "/content/DroneAudioDataset/Multiclass_Drone_Audio/bg noise"
        try:
            if is_threat:
                all_drone_files = []
                for d in DRONE_DIRS:
                    all_drone_files.extend([os.path.join(d, f) for f in os.listdir(d) if f.endswith('.wav')])
                source_file = random.choice(all_drone_files)
            else:
                all_bg_files = [os.path.join(AIRPORT_DIR, f) for f in os.listdir(AIRPORT_DIR) if f.endswith('.wav')]
                source_file = random.choice(all_bg_files)
            y, _ = librosa.load(source_file, sr=SR, duration=1.0)
            audio_confidence = get_prediction_confidence(y, SR)
        except Exception:
            audio_confidence = 0.0
    audio_detected = audio_confidence > 70

    # --- Vision inference ---
    vision_confidence = 0.0
    try:
        _DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        if _cached_vision_model is not None:
            demo_dir = Path('/content/demo_images')
            selected_category = random.choice(['birds', 'drones', 'planes'])
            all_images = list((demo_dir / selected_category).glob('*.jpg'))
            if all_images:
                img_path = random.choice(all_images)
                transform = transforms.Compose([
                    transforms.Resize((224, 224)),
                    transforms.ToTensor(),
                    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
                ])
                img = Image.open(img_path).convert('RGB')
                input_tensor = transform(img).unsqueeze(0).to(_DEVICE)
                with torch.no_grad():
                    output = _cached_vision_model(input_tensor)
                    vision_confidence = torch.sigmoid(output).item() * 100
    except Exception:
        vision_confidence = 0.0
        
    vision_detected = vision_confidence > 50
    rf_confidence = 100.0 if any_triggered else 0.0
    overall_confidence = (rf_confidence + audio_confidence + vision_confidence) / 3
    votes = sum([any_triggered, audio_detected, vision_detected])
    drone_present = audio_detected or vision_detected
    signal_threat = any_triggered
    overall_threat = drone_present and signal_threat

    sys_state.latest_snapshot = {
        "timestamp": time.time(),
        "drone_pos": CURRENT_POS,
        "is_threat": is_threat,
        "rf": {
            "triggered_sensors": sys_state.cur_analysis,
            "detected": any_triggered
        },
        "audio": {
            "confidence": audio_confidence,
            "detected": audio_detected
        },
        "vision": {
            "confidence": vision_confidence,
            "detected": vision_detected
        },
        "fused": {
            "drone_present": drone_present,
            "signal_threat": signal_threat,
            "overall_threat": overall_threat,
            "overall_confidence": overall_confidence,
            "votes": votes
        }
    }

    return {
        "p1": p1,
        "p2": p2,
        "dt": dt,
        "is_threat": is_threat,
        "sensor_states": sensor_states,
    }

def build_status_circle_html():
    movement = drone_movement_and_detection()
    movement_duration = max(0.05, movement["dt"])

    p1 = movement["p1"]
    p2 = movement["p2"]
    is_threat = movement["is_threat"]
    sensor_states = movement["sensor_states"]

    animation_name = f"moveDot{random.randint(0, 100000)}"

    img_src = ""
    if os.path.exists(IMAGE_PATH):
        with open(IMAGE_PATH, "rb") as f:
            encoded = base64.b64encode(f.read()).decode("utf-8")
        ext = IMAGE_PATH.split(".")[-1].lower()
        if ext == "jpg":
            ext = "jpeg"
        img_src = f"data:image/{ext};base64,{encoded}"

    drone_img_src = load_drone_as_data_uri(DRONE_IMAGE_PATH)

    drone_filter = (
      "brightness(0) invert(0.6)" if not is_threat else
      "brightness(0) saturate(100%) invert(20%) sepia(100%) saturate(6000%) hue-rotate(0deg) brightness(1.2)"
    )

    sensor_html = []
    for sensor in sensor_states:
        sx = sensor["x"]
        sy = sensor["y"]
        triggered = sensor["triggered"]

        sensor_color = "rgba(255, 165, 0, 0.16)" if triggered else "rgba(100, 180, 255, 0.10)"
        sensor_border = "rgba(255, 180, 80, 0.40)" if triggered else "rgba(120, 190, 255, 0.22)"

        sensor_html.append(f"""
            <div style="position: absolute; left: {sx}px; top: {sy}px; width: {2 * SENSOR_RADIUS_PX}px; height: {2 * SENSOR_RADIUS_PX}px; transform: translate(-50%, -50%); border-radius: 50%; background: {sensor_color}; border: 1.5px solid {sensor_border}; box-shadow: 0 0 12px {sensor_color}; pointer-events: none;"></div>
        """)

    sensors_html = "\n".join(sensor_html)

    return f"""
    <style>
    @keyframes {animation_name} {{
        0%   {{ left: {p1[0]}px; top: {p1[1]}px; }}
        100% {{ left: {p2[0]}px; top: {p2[1]}px; }}
    }}
    </style>

    <div style="height: 100%; min-height: {IMAGE_HEIGHT_PX + 32}px; display: flex; align-items: center; justify-content: center; background: #0f1117; border-radius: 18px; border: 1px solid #2a2f3a; padding: 16px; box-sizing: border-box;">
        <div style="position: relative; width: {IMAGE_WIDTH_PX}px; height: {IMAGE_HEIGHT_PX}px; min-width: {IMAGE_WIDTH_PX}px; max-width: {IMAGE_WIDTH_PX}px; min-height: {IMAGE_HEIGHT_PX}px; max-height: {IMAGE_HEIGHT_PX}px; overflow: hidden; border-radius: 18px; flex-shrink: 0;">
            <img src="{img_src}" style="width: {IMAGE_WIDTH_PX}px; height: {IMAGE_HEIGHT_PX}px; min-width: {IMAGE_WIDTH_PX}px; max-width: {IMAGE_WIDTH_PX}px; min-height: {IMAGE_HEIGHT_PX}px; max-height: {IMAGE_HEIGHT_PX}px; object-fit: fill; display: block; border-radius: 18px;">
            {sensors_html}
            <img src="{drone_img_src}" style="position: absolute; left: {p1[0]}px; top: {p1[1]}px; width: {2 * DRONE_RADIUS_PX * 2}px; height: {2 * DRONE_RADIUS_PX * 2}px; transform: translate(-50%, -50%); animation: {animation_name} {movement_duration}s linear forwards; filter: {drone_filter}; opacity: 0.95; pointer-events: none;">
        </div>
    </div>
    """

def build_blank_visual_panel():
    return """
    <div style="width: 100%; aspect-ratio: 1 / 1; min-height: 320px; background: #0f1117; border: 2px dashed #3b4252; border-radius: 18px; display: flex; align-items: center; justify-content: center; color: #8b949e; font-size: 20px; font-weight: 600;">
        Visualization Output
    </div>
    """

# ---- CHATBOT -----
nest_asyncio.apply()

mcp = FastMCP("RF_Analysis_Assistant", port=8001)


@mcp.tool()
def check_current_rf_status() -> str:
    """Reads the current RF environment variables including modulation and frequency hopping."""
    if sys_state.cur_analysis:
        # Prevent type error: cur_analysis is assigned as a list of integers during movement
        if isinstance(sys_state.cur_analysis, list):
          return f"Anomalous RF transmissions detected at sensor indices: {sys_state.cur_analysis}"
    return "No RF analysis available. The operator has not scanned a signal yet."




# Audio Tool begins ==========================================================================================================

@mcp.tool()
def check_current_audio_status() -> str:
    """Checks audio signals to detect whether a drone is present and prepares audio samples."""
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    SR = 22050
    SENSOR_POSITIONS = SENSORS
    IS_DRONE_RANGE = IS_DRONE_AUDIO_RANGE
    AIRPORT_DIR = "/content/DroneAudioDataset/Multiclass_Drone_Audio/bg noise"
    DRONE_DIRS = [
        "/content/DroneAudioDataset/Multiclass_Drone_Audio/membo_1",
        "/content/DroneAudioDataset/Multiclass_Drone_Audio/bebop_1"
    ]

    sensor_data = []
    nearest = []

    for i, s_pos in enumerate(SENSOR_POSITIONS):
        dist = np.sqrt((CURRENT_POS[0] - s_pos[0])**2 + (CURRENT_POS[1] - s_pos[1])**2)

        # Load Signal
        if IS_DRONE_RANGE:
            all_drone_files = []
            for d in DRONE_DIRS:
                all_drone_files.extend([os.path.join(d, f) for f in os.listdir(d) if f.endswith('.wav')])
            source_file = random.choice(all_drone_files)
            y, _ = librosa.load(source_file, sr=SR, duration=2.0) # 2s for better playback
            attenuated_y = y * (1.0 / (dist + 1e-6))
        else:
            all_bg_files = [os.path.join(AIRPORT_DIR, f) for f in os.listdir(AIRPORT_DIR) if f.endswith('.wav')]
            source_file = random.choice(all_bg_files)
            attenuated_y, _ = librosa.load(source_file, sr=SR, duration=2.0)

        conf = get_prediction_confidence(attenuated_y, SR)
        sensor_data.append({
            "idx": i,
            "dist": dist,
            "conf": conf,
            "audio": attenuated_y
        })

    # Sort by distance and pick top 1
    nearest = sorted(sensor_data, key=lambda x: x['dist'])[:1]

    # Save audio files and store base64 in sys_state for the UI
    sys_state.recent_audio_html = ""
    audio_elements = []

    for item in nearest:
        # Convert to base64 for direct HTML embedding (prevents path issues in Colab/Gradio)
        buf = io.BytesIO()
        sf.write(buf, item["audio"], SR, format='WAV')
        b64_audio = base64.b64encode(buf.getvalue()).decode("utf-8")

        audio_elements.append(f"""
            <div style="margin-bottom: 10px; padding: 8px; background: #1e222c; border-radius: 8px; border-left: 4px solid #00CCFF;">
                <p style="margin: 0 0 5px 0; font-size: 12px; color: #8b949e;">Sensor {item['idx']} (Dist: {item['dist']:.1f}px | Conf: {item['conf']:.1f}%)</p>
                <audio controls style="height: 30px; width: 100%;">
                    <source src="data:audio/wav;base64,{b64_audio}" type="audio/wav">
                </audio>
            </div>
        """)

    sys_state.recent_audio_html = "".join(audio_elements)
    try:
        audio_data = nearest[0]['audio']
        spec = librosa.feature.melspectogram(y = audio_data, sr = SR, n_mels = 128)
        spec_db = librosa.power_to_db(spec, ref=np.max)
        
    if nearest[0]['conf'] > 70:
        return f"Acoustic analysis complete. Drone detected at closest sensor {nearest[0]['idx']}."
    return "Acoustic analysis complete. Drone not detected."
# Audio Tool ends ===============================================================================================================

# Vision Tool begins ============================================================================================================

_cached_vision_model = None
global_visual_html = None

@mcp.tool()
def check_current_visual_status() -> str:
    """
    Randomly samples an image from the visual sensor feeds, runs the ResNet50
    vision model, and returns whether a drone is present or not.
    Use this tool whenever the user asks to check cameras or look for drones visually.
    """
    global _cached_vision_model, global_visual_html
    _DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Initialize Model (Only runs once)
    if _cached_vision_model is None:
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, 1)
        # MUST MATCH YOUR SAVED WEIGHTS PATH
        checkpoint = torch.load('/content/drive/MyDrive/ARL/resnet50_drone_weights.pth', map_location=_DEVICE)

        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            model.load_state_dict(checkpoint)

        model.to(_DEVICE)
        model.eval()
        _cached_vision_model = model

    # 2. Pick Random Image
    demo_dir = Path('/content/demo_images')
    selected_category = random.choice(['birds', 'drones', 'planes'])
    all_images = list((demo_dir / selected_category).glob('*.jpg'))

    if not all_images:
        return "Error: Camera feed offline. No images found."

    img_path = random.choice(all_images)

    # 3. Preprocess & Infer
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    img = Image.open(img_path).convert('RGB')
    input_tensor = transform(img).unsqueeze(0).to(_DEVICE)

    with torch.no_grad():
        output = _cached_vision_model(input_tensor)
        prob = torch.sigmoid(output).item()

    is_drone = prob >= 0.5
    classification = "DRONE DETECTED" if is_drone else "CLEAR (Bird/Plane)"
    color = "#ef4444" if is_drone else "#22c55e" # Red for drone, Green for clear

    # 4. Generate the UI HTML secretly for Gradio
    buffered = BytesIO()
    img.save(buffered, format="JPEG")
    img_str = base64.b64encode(buffered.getvalue()).decode()

    global_visual_html = f"""
    <div style="position: relative; width: 100%; aspect-ratio: 1/1; background: #0f1117; border-radius: 18px; overflow: hidden; border: 2px solid {color};">
        <div style="position: absolute; top: 0; left: 0; right: 0; background: rgba(0,0,0,0.8); color: {color}; padding: 8px; text-align: center; font-family: monospace; font-weight: bold; z-index: 10;">
            OPTICAL SENSOR: {classification} ({prob:.1%} Conf)
        </div>
        <img src="data:image/jpeg;base64,{img_str}" style="width: 100%; height: 100%; object-fit: contain; padding-top: 35px;">
    </div>
    """

    # 5. Return text to the Chatbot Agent
    return f"Visual Analysis Complete. Result: {classification}. Confidence: {prob:.4f}."

# Vision Tool Ends =======================================================================================================


@mcp.tool()
def get_aggregate() -> str:
    """Use this tool when the user asks about overall threat status or wants a combined assessment from all sensors. Returns fused RF, audio, and vision results."""
    if sys_state.latest_snapshot["timestamp"] is None:
        return "No data available yet. System has not started yet."
    snap = sys_state.latest_snapshot
    return f"RF Data: Detected - {snap['rf']['detected']}, Sensors Triggered - {snap['rf']['triggered_sensors']}, Audio: Detected - {snap['audio']['detected']}, Confidence - {snap['audio']['confidence']}%, Vision: Detected - {snap['vision']['detected']}, Confidence - {snap['vision']['confidence']}%, Drone Present - {snap['fused']['drone_present']}, Signal Threat - {snap['fused']['signal_threat']}, Overall Threat - {snap['fused']['overall_threat']}, Confidence - {snap['fused']['overall_confidence']}%, Votes - {snap['fused']['votes']}/3"


def start_server():
    print("Starting MCP Server...")
    mcp.run(transport="sse")

server_thread = threading.Thread(target=start_server, name="mcp_server_thread", daemon=True)
server_thread.start()

!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

def run_ollama():
    subprocess.run(["ollama", "serve"])

ollama_thread = threading.Thread(target=run_ollama, daemon=True)
ollama_thread.start()

print("✅ Ollama server is running!")

# Wait for Ollama server to start
for _ in range(30): # Try for 30 seconds
    try:
        subprocess.run(["ollama", "list"], check=True, capture_output=True, timeout=1)
        print("Ollama server is responsive.")
        break
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        print("Waiting for Ollama server...")
        time.sleep(1)
else:
    print("Ollama server did not become responsive in time.")

!ollama pull llama3.2

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

SYSTEM_PROMPT = (
    "You are an RF engineering assistant in the US army. "
    "You have access to three tools:\n"
    "1. `check_current_rf_status`: Analyzes and provides sensors where anomalous rf transmission were detected.\n"
    "2. `check_current_audio_status`: Checks audio signals to detect whether a drone is present. Use this whenever the user asks about audio, acoustics, or drone presence.\n"
    "3. `check_current_visual_status`: Takes a picture/photo of the current environment using a camera, and identifies whether a drone is present or not."
    "4. `get_aggregate`: Checks all three streams of data to aggregate the results and determine whether a drone is present and whether it is a threat or not."
    "Based on the results from these tools, your task is to answer the questions so that even a civilian would understand what is going on."
)

global_agent = None

async def get_agent():
    global global_agent
    if global_agent is None:
        client = MultiServerMCPClient({
            "research": {
                "transport": "sse",
                "url": "http://localhost:8000/sse"
            }
        })
        mcp_tools = await client.get_tools()
        global_agent = create_react_agent(llm, mcp_tools)
    return global_agent

async def chat_fn(message, history):
    if not message:
        return "Please enter a question."

    try:
        agent = await get_agent()
        messages = [SystemMessage(content=SYSTEM_PROMPT)]

        if history:
            for msg in history:
                if msg.get("role") == "user":
                    messages.append(HumanMessage(content=msg.get("content", "")))
                elif msg.get("role") == "assistant":
                    messages.append(AIMessage(content=msg.get("content", "")))

        messages.append(HumanMessage(content=message))
        response = await agent.ainvoke({"messages": messages})
        return response["messages"][-1].content

    except Exception as e:
        return f"Error connecting to agent: {str(e)}"

async def respond(message, chat_history):
    if chat_history is None:
        chat_history = []

    # Initialize with a blank panel
    vis_html = build_blank_visual_panel()
    global global_visual_html

    try:
        agent = await get_agent()

        # Prepare Message History for LangChain/MCP
        messages = [SystemMessage(content=SYSTEM_PROMPT)]
        for msg in chat_history:
            messages.append(
                AIMessage(content=msg["content"]) if msg["role"] == "assistant"
                else HumanMessage(content=msg["content"])
            )
        messages.append(HumanMessage(content=message))

        # 1. EXECUTE AGENT
        # The agent will decide to call either the Audio, Visual, or RF tool
        response = await agent.ainvoke({"messages": messages})
        bot_message = response["messages"][-1].content

        # 2. UI ACTIVATION LOGIC
        # We check which "sensor" left data in the state/globals

        # CASE A: Audio Tool was called
        if hasattr(sys_state, 'recent_audio_html') and sys_state.recent_audio_html:
            vis_html = f"""
            <div style="width: 100%; max-height: 400px; overflow-y: auto; background: #0f1117; border-radius: 18px; padding: 15px; border: 1px solid #2a2f3a;">
                <h3 style="color: white; margin-top: 0;">🔊 Nearest Sensor Audio</h3>
                {sys_state.recent_audio_html}
            </div>
            """
            sys_state.recent_audio_html = None # Clear after display

        # CASE B: Visual Tool was called
        elif global_visual_html:
            vis_html = global_visual_html
            global_visual_html = None  # Clear for next turn

        # CASE C: RF/Spectrogram Tool was called (Fallback/Specific)
        elif hasattr(sys_state, 'cur_spec_base64') and sys_state.cur_spec_base64:
            vis_html = f"""
            <div style="width: 100%; aspect-ratio: 1/1; background: #0f1117; border-radius: 18px; overflow: hidden; border: 1px solid #2a2f3a;">
                <img src="data:image/png;base64,{sys_state.cur_spec_base64}" style="width: 100%; height: 100%; object-fit: contain;">
            </div>
            """
            sys_state.cur_spec_base64 = None  # Clear for next turn

    except Exception as e:
        bot_message = f"Agent Error: {str(e)}"
        vis_html = build_blank_visual_panel()

    # Update history and return to UI
    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": bot_message})

    return "", chat_history, vis_html

# --- UI CONTROL FUNCTIONS ---
def refresh_ui(noise):
    status_html = build_status_circle_html()
    visual_html = build_blank_visual_panel()
    return status_html, visual_html, "Monitoring..."

def stream(noise):
    sys_state.running = True
    while sys_state.running:
        status_html = build_status_circle_html()
        visual_html = build_blank_visual_panel()
        yield status_html, visual_html, "Monitoring..."
        time.sleep(2.0)

def pause():
    sys_state.running = False
    return "Paused"

# --- INTERFACE ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📡 RF Interceptor Dashboard")

    with gr.Row():
        with gr.Column(scale=1):
            status_circle = gr.HTML(build_status_circle_html())

            noise = gr.Slider(0, 0.5, 0.1, label="Noise")
            btn_play = gr.Button("▶ Start", variant="primary")
            btn_pause = gr.Button("⏸ Pause", variant="secondary")
            out = gr.HTML("Ready")

        with gr.Column(scale=1):
            chatbot = gr.Chatbot(label="ChatBot", height=320)
            msg = gr.Textbox(label="Your Message")
            clear = gr.Button("Clear Memory")
            visual_panel = gr.HTML(build_blank_visual_panel())

    btn_play.click(stream, [noise], [status_circle, visual_panel, out])
    btn_pause.click(pause, None, out)

    # Fixed outputs to match the three variables returned by respond
    msg.submit(respond, inputs=[msg, chatbot], outputs=[msg, chatbot, visual_panel])
    clear.click(lambda: [], None, chatbot, queue=False)

if __name__ == "__main__":
    demo.queue().launch(debug=True)

In [ ]:
!fuser -k 8001/tcp